# Airline Sentiment Classifier - Inference Only
**Author:** Rizwan  
**Model Type:** RNN (Bidirectional LSTM)  
**Purpose:** Load pre-trained model for predictions (NO TRAINING)

This notebook loads the trained model and makes predictions without retraining.

Quick Start:
1. Run all cells in order
2. Model loads in about 5 seconds
3. Ready for predictions

## 1. Setup & Imports

In [1]:
# Basic imports
import os
import pickle
import re
print("Step 1: Basic imports loaded")

# NumPy and Pandas
import numpy as np
import pandas as pd
print("Step 2: NumPy and Pandas loaded")

# PyTorch
import torch
import torch.nn as nn
print("Step 3: PyTorch loaded")

# Configuration
torch.manual_seed(42)
np.random.seed(42)

import warnings
warnings.filterwarnings('ignore')

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nPyTorch Version: {torch.__version__}")
print(f"Device: {device}")
print("\nAll imports complete!")

Step 1: Basic imports loaded
Step 2: NumPy and Pandas loaded
Step 3: PyTorch loaded

PyTorch Version: 2.7.1+cu118
Device: cpu

All imports complete!


## 2. Configure Paths

In [3]:
# Configure paths relative to project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
MODELS_PATH = os.path.join(PROJECT_ROOT, 'models', 'RNN')

print(f"Loading models from: {MODELS_PATH}")

# Check if models exist
model_path = os.path.join(MODELS_PATH, 'sentiment_classifier_final.pt')
vocab_path = os.path.join(MODELS_PATH, 'vocab.pkl')

if not os.path.exists(model_path):
    print(f"\nWARNING: Model not found at: {model_path}")
    print("Please run the training notebook first (Rizwan_doc.ipynb)")
else:
    print(f"Model found: {os.path.basename(model_path)}")
    
if not os.path.exists(vocab_path):
    print(f"\nWARNING: Vocabulary not found at: {vocab_path}")
    print("Please run the training notebook first (Rizwan_doc.ipynb)")
else:
    print(f"Vocabulary found: {os.path.basename(vocab_path)}")

Loading models from: c:\Users\rizwa\OneDrive\Desktop\NP IT\year 2 sem 2\CVNL\CVNL_ASSG_Team4\models\RNN
Model found: sentiment_classifier_final.pt
Vocabulary found: vocab.pkl


## 3. Load Model Architecture

In [4]:
class SentimentLSTM(nn.Module):
    """
    Bidirectional LSTM for sentiment classification.
    Same architecture as used during training.
    """
    def __init__(self, vocab_size, embedding_dim=150, hidden_dim=160,
                 output_dim=3, n_layers=3, bidirectional=True, dropout=0.4):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            dropout=dropout if n_layers > 1 else 0,
            batch_first=True
        )
        
        lstm_output_dim = hidden_dim * 2 if bidirectional else hidden_dim
        
        self.fc1 = nn.Linear(lstm_output_dim, 64)
        self.fc2 = nn.Linear(64, output_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, text):
        embedded = self.dropout(self.embedding(text))
        lstm_out, (hidden, cell) = self.lstm(embedded)
        hidden = self.dropout(hidden)
        
        if self.lstm.bidirectional:
            hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        else:
            hidden = hidden[-1]
        
        out = self.fc1(hidden)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out

print("Model architecture loaded")

Model architecture loaded


## 4. Load Vocabulary

In [5]:
# Vocabulary class (needed for loading pickle file)
class Vocabulary:
    def __init__(self, min_freq=2):
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        self.word_freq = {}
        self.min_freq = min_freq

    def build_vocab(self, texts):
        for text in texts:
            for word in text.split():
                self.word_freq[word] = self.word_freq.get(word, 0) + 1
        idx = 2
        for word, freq in self.word_freq.items():
            if freq >= self.min_freq:
                self.word2idx[word] = idx
                self.idx2word[idx] = word
                idx += 1

    def encode(self, text):
        return [self.word2idx.get(word, 1) for word in text.split()]

print("Vocabulary class defined")

Vocabulary class defined


In [6]:
# Load the vocabulary built during training
with open(vocab_path, 'rb') as f:
    vocab = pickle.load(f)

vocab_size = len(vocab.word2idx)
print(f"Vocabulary loaded")
print(f"Vocabulary size: {vocab_size:,} words")

Vocabulary loaded
Vocabulary size: 5,228 words


## 5. Load Pre-Trained Model

In [7]:
print("="*80)
print("LOADING PRE-TRAINED MODEL")
print("="*80)

# Create model with same architecture as training (Iteration 5)
model = SentimentLSTM(
    vocab_size=vocab_size,
    embedding_dim=150,
    hidden_dim=160,
    output_dim=3,
    n_layers=3,
    bidirectional=True,
    dropout=0.4
).to(device)

# Load trained weights
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()  # Set to evaluation mode

print(f"\nPre-trained model loaded successfully")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {device}")
print(f"Model file: {os.path.basename(model_path)}")
print("\nReady for inference")
print("="*80)

LOADING PRE-TRAINED MODEL

Pre-trained model loaded successfully
Total parameters: 2,438,219
Device: cpu
Model file: sentiment_classifier_final.pt

Ready for inference


## 6. Text Preprocessing

In [8]:
def clean_text(text):
    """
    Clean and preprocess text (same as training).
    """
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)  # Remove URLs
    text = re.sub(r'@\w+', '', text)  # Remove mentions
    text = re.sub(r'#', '', text)  # Remove hashtag symbol
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special chars
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra whitespace
    return text

print("Text preprocessing function ready")

Text preprocessing function ready


## 7. Prediction Function

In [9]:
def predict_sentiment(text, return_probabilities=False):
    """
    Predict sentiment for a given text.
    
    Args:
        text (str): Input text to classify
        return_probabilities (bool): If True, return confidence scores
    
    Returns:
        str or dict: Predicted sentiment ('positive', 'negative', 'neutral')
                     or dict with sentiment and probabilities
    """
    # Preprocessing
    cleaned_text = clean_text(text)
    encoded = vocab.encode(cleaned_text)
    
    # Pad/truncate to fixed length (50 tokens as in training)
    max_len = 50
    if len(encoded) < max_len:
        encoded = encoded + [0] * (max_len - len(encoded))
    else:
        encoded = encoded[:max_len]
    
    # Convert to tensor
    text_tensor = torch.tensor([encoded], dtype=torch.long).to(device)
    
    # Prediction
    with torch.no_grad():
        output = model(text_tensor)
        probabilities = torch.softmax(output, dim=1)
        predicted_class = output.argmax(dim=1).item()
    
    # Label mapping
    label_map = {0: 'positive', 1: 'negative', 2: 'neutral'}
    sentiment = label_map[predicted_class]
    
    if return_probabilities:
        return {
            'sentiment': sentiment,
            'confidence': probabilities[0][predicted_class].item(),
            'probabilities': {
                'positive': probabilities[0][0].item(),
                'negative': probabilities[0][1].item(),
                'neutral': probabilities[0][2].item()
            }
        }
    else:
        return sentiment

print("Prediction function ready")

Prediction function ready


## 8. Batch Prediction Function

In [10]:
def predict_batch(texts, batch_size=32):
    """
    Predict sentiments for multiple texts efficiently.
    
    Args:
        texts (list): List of text strings
        batch_size (int): Batch size for processing
    
    Returns:
        list: List of predicted sentiments
    """
    predictions = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        
        # Preprocess batch
        batch_encoded = []
        for text in batch_texts:
            cleaned = clean_text(text)
            encoded = vocab.encode(cleaned)
            
            if len(encoded) < 50:
                encoded = encoded + [0] * (50 - len(encoded))
            else:
                encoded = encoded[:50]
            
            batch_encoded.append(encoded)
        
        # Convert to tensor
        batch_tensor = torch.tensor(batch_encoded, dtype=torch.long).to(device)
        
        # Predict
        with torch.no_grad():
            outputs = model(batch_tensor)
            pred_classes = outputs.argmax(dim=1).cpu().numpy()
        
        # Convert to labels
        label_map = {0: 'positive', 1: 'negative', 2: 'neutral'}
        batch_predictions = [label_map[pred] for pred in pred_classes]
        predictions.extend(batch_predictions)
    
    return predictions

print("Batch prediction function ready")

Batch prediction function ready


## 9. Test Predictions

In [11]:
print("\n" + "="*80)
print("TESTING MODEL WITH SAMPLE INPUTS")
print("="*80)

# Test examples
test_examples = [
    "This flight was amazing! Best service ever!",
    "Worst airline experience of my life. Never flying with them again.",
    "The flight was okay, nothing special.",
    "@AmericanAir you lost my luggage AGAIN! Unbelievable!",
    "Thank you for the upgrade! Very comfortable flight 😊"
]

print("\nSingle predictions with probabilities:\n")
for text in test_examples:
    result = predict_sentiment(text, return_probabilities=True)
    print(f"Text: \"{text}\"")
    print(f"  → Predicted: {result['sentiment'].upper()} (confidence: {result['confidence']:.2%})")
    print(f"     Pos: {result['probabilities']['positive']:.2%} | "
          f"Neg: {result['probabilities']['negative']:.2%} | "
          f"Neu: {result['probabilities']['neutral']:.2%}")
    print()

print("\n" + "="*80)
print("Batch prediction (faster for multiple texts):")
print("="*80)

batch_results = predict_batch(test_examples)
for text, pred in zip(test_examples, batch_results):
    print(f"  [{pred.upper()}] {text}")


TESTING MODEL WITH SAMPLE INPUTS

Single predictions with probabilities:

Text: "This flight was amazing! Best service ever!"
  → Predicted: POSITIVE (confidence: 98.46%)
     Pos: 98.46% | Neg: 0.33% | Neu: 1.21%

Text: "Worst airline experience of my life. Never flying with them again."
  → Predicted: NEGATIVE (confidence: 99.67%)
     Pos: 0.07% | Neg: 99.67% | Neu: 0.26%

Text: "The flight was okay, nothing special."
  → Predicted: NEGATIVE (confidence: 97.40%)
     Pos: 0.76% | Neg: 97.40% | Neu: 1.84%

Text: "@AmericanAir you lost my luggage AGAIN! Unbelievable!"
  → Predicted: NEGATIVE (confidence: 99.67%)
     Pos: 0.06% | Neg: 99.67% | Neu: 0.27%

Text: "Thank you for the upgrade! Very comfortable flight 😊"
  → Predicted: POSITIVE (confidence: 97.86%)
     Pos: 97.86% | Neg: 0.57% | Neu: 1.57%


Batch prediction (faster for multiple texts):
  [POSITIVE] This flight was amazing! Best service ever!
  [NEGATIVE] Worst airline experience of my life. Never flying with them again.


## 10. Integration Example: CSV File Processing

In [12]:
def process_csv_file(input_csv_path, text_column='text', output_csv_path=None):
    """
    Process a CSV file and add sentiment predictions.
    
    Args:
        input_csv_path (str): Path to input CSV file
        text_column (str): Name of column containing text
        output_csv_path (str): Path to save results (optional)
    
    Returns:
        pd.DataFrame: DataFrame with predictions added
    """
    # Load CSV
    df = pd.read_csv(input_csv_path)
    
    if text_column not in df.columns:
        raise ValueError(f"Column '{text_column}' not found in CSV. Available: {df.columns.tolist()}")
    
    print(f"Processing {len(df)} rows...")
    
    # Predict sentiments
    texts = df[text_column].fillna('').tolist()
    predictions = predict_batch(texts)
    
    # Add predictions to dataframe
    df['predicted_sentiment'] = predictions
    
    # Save if output path provided
    if output_csv_path:
        df.to_csv(output_csv_path, index=False)
        print(f"Results saved to: {output_csv_path}")
    
    return df

print("CSV processing function ready")
print("\nUsage example:")
print("  df = process_csv_file('tweets.csv', text_column='tweet_text', output_csv_path='tweets_with_sentiment.csv')")

CSV processing function ready

Usage example:
  df = process_csv_file('tweets.csv', text_column='tweet_text', output_csv_path='tweets_with_sentiment.csv')


## 11. Interactive Demo

In [13]:
# Interactive prediction
print("\n" + "="*80)
print("INTERACTIVE SENTIMENT ANALYSIS")
print("="*80)
print("Enter text to analyze (or press Enter to skip):\n")

user_input = input("Your text: ").strip()

if user_input:
    result = predict_sentiment(user_input, return_probabilities=True)
    
    print(f"\n{'='*80}")
    print(f"INPUT: \"{user_input}\"")
    print(f"{'='*80}")
    print(f"Predicted Sentiment: {result['sentiment'].upper()}")
    print(f"Confidence: {result['confidence']:.2%}")
    print(f"\nProbability Breakdown:")
    print(f"  • Positive: {result['probabilities']['positive']:.2%}")
    print(f"  • Negative: {result['probabilities']['negative']:.2%}")
    print(f"  • Neutral:  {result['probabilities']['neutral']:.2%}")
    print(f"{'='*80}")
else:
    print("Skipped interactive demo")


INTERACTIVE SENTIMENT ANALYSIS
Enter text to analyze (or press Enter to skip):


INPUT: "wow i loved it not"
Predicted Sentiment: NEGATIVE
Confidence: 62.08%

Probability Breakdown:
  • Positive: 22.79%
  • Negative: 62.08%
  • Neutral:  15.14%


## 12. Model Information

In [14]:
print("\n" + "="*80)
print("MODEL INFORMATION")
print("="*80)

print(f"\nModel Architecture:")
print(f"  Type: Bidirectional LSTM")
print(f"  Embedding Dimension: 150")
print(f"  Hidden Dimension: 160")
print(f"  LSTM Layers: 3")
print(f"  Dropout: 0.4")
print(f"  Total Parameters: {sum(p.numel() for p in model.parameters()):,}")

print(f"\nVocabulary:")
print(f"  Size: {vocab_size:,} unique tokens")
print(f"  Special tokens: <PAD> (0), <UNK> (1)")

print(f"\nOutput Classes:")
print(f"  Positive (0)")
print(f"  Negative (1)")
print(f"  Neutral (2)")

print(f"\nModel Files:")
print(f"  Weights: {os.path.basename(model_path)}")
print(f"  Vocabulary: {os.path.basename(vocab_path)}")
print(f"  Location: {MODELS_PATH}")

print(f"\nDevice: {device}")

print(f"\n{'='*80}")
print("MODEL READY FOR USE")
print("="*80)


MODEL INFORMATION

Model Architecture:
  Type: Bidirectional LSTM
  Embedding Dimension: 150
  Hidden Dimension: 160
  LSTM Layers: 3
  Dropout: 0.4
  Total Parameters: 2,438,219

Vocabulary:
  Size: 5,228 unique tokens
  Special tokens: <PAD> (0), <UNK> (1)

Output Classes:
  Positive (0)
  Negative (1)
  Neutral (2)

Model Files:
  Weights: sentiment_classifier_final.pt
  Vocabulary: vocab.pkl
  Location: c:\Users\rizwa\OneDrive\Desktop\NP IT\year 2 sem 2\CVNL\CVNL_ASSG_Team4\models\RNN

Device: cpu

MODEL READY FOR USE


---

## Notes for Integration:

### Quick Start Functions:
- **`predict_sentiment(text)`** - Single prediction
- **`predict_batch(texts)`** - Batch prediction (faster)
- **`process_csv_file(path)`** - Process entire CSV files

### Example Integration:
```python
# Simple usage
sentiment = predict_sentiment("Great flight!")  # Returns: 'positive'

# With confidence scores
result = predict_sentiment("Great flight!", return_probabilities=True)
# Returns: {'sentiment': 'positive', 'confidence': 0.95, 'probabilities': {...}}

# Batch processing
sentiments = predict_batch(["Text 1", "Text 2", "Text 3"])
```

### Performance:
- Single prediction: ~0.01 seconds
- Batch (100 texts): ~0.5 seconds
- No retraining needed
- Models cached in memory

---